# Pestaña 3 · Modelos base

La tercera pestaña evalúa los modelos base en el periodo futuro reservado para la prueba. Son dos: una regla simple que predice siempre «No» (un `DummyClassifier` con la prevalencia del entrenamiento) y una regresión logística con `C = 100`, que se evalúa con el umbral estándar de 0,5 y con un umbral ajustado en la validación temporal del entrenamiento. El enunciado permite elegir entre regresión logística o SVR lineal; se elige la logística porque el objetivo es categórico (el cliente suscribe o no) y, por tanto, se trata de clasificación.

Esta pestaña no responde al selector de año, porque la prueba temporal debe permanecer fija. Cada interpretación sigue el orden de siempre: lo que muestra el resultado, lo que significa para el modelo y la decisión que se toma.

![Vista de la pestaña 3 del dashboard](../imagenes/pestana_3_modelos.png)

La pestaña contiene seis resultados:

1. Las tarjetas de desempeño: AUC-ROC, AUC-PR, error de Brier y suscripciones reales.
2. La tabla de métricas en prueba de los tres modelos.
3. La curva ROC.
4. Las matrices de confusión con el umbral de 0,5 y con el umbral ajustado.
5. La lectura del punto operativo.
6. La configuración del modelo.

In [1]:
import sys
sys.path.insert(0, "libro_herramientas")
from informe_dashboard import *
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss, confusion_matrix,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import TimeSeriesSplit

original, clean, r = cargar()
tr, te, y_tr, y_te = r["train"], r["test"], r["y_train"], r["y_test"]
p, umbral = r["probability"], r["threshold"]
prev = float(y_te.mean())
comp = componentes(d.make_model_tab(r))               # figuras y tarjetas tal como las arma el dashboard

modelos = {
    "Regla simple · siempre «No»": (r["dummy_probability"], r["dummy_prediction"]),
    "Regresión logística · umbral 0,5": (p, r["prediction_05"]),
    f"Regresión logística · umbral ajustado {num(umbral, 3)}": (p, r["prediction_tuned"]),
}
metricas = pd.DataFrame({nombre: d.metric_row(y_te, s, q) for nombre, (s, q) in modelos.items()}).T
metricas.columns = ["Exactitud", "Precisión", "Recall", "F1", "AUC-ROC", "AUC-PR", "Brier"]
m0, m05, mu = (metricas.iloc[i] for i in range(3))

## 3.1 Tarjetas de desempeño

In [2]:
tabla(tabla_de_tarjetas(comp["tarjetas"]).rename_axis(None), dec=0)

,Valor,Nota
AUC-ROC,"0,677","1 es perfecto; 0,5 equivale al azar"
AUC precisión-recall,"0,489","Referencia de la prueba: 30,8 %"
Error de probabilidad (Brier),"0,215",Más bajo indica probabilidades más precisas
Suscripciones reales,2.539,De 8.236 contactos de prueba


In [3]:
auc = roc_auc_score(y_te, p)
ap = average_precision_score(y_te, p)
brier = brier_score_loss(y_te, p)
brier0 = brier_score_loss(y_te, r["dummy_probability"])
interpretar(
    f"En la prueba, el AUC-ROC es {num(auc, 3)}, el AUC-PR es {num(ap, 3)} frente a una prevalencia de "
    f"{num(prev, 3)} ({num(ap / prev, 1)} veces) y el error de Brier es {num(brier, 3)} frente a {num(brier0, 3)} "
    f"de la regla simple, una mejora relativa de {pct(1 - brier / brier0)}. Entre los {num(len(y_te), 0)} contactos "
    f"de prueba hay {num(int(y_te.sum()), 0)} suscripciones reales ({pct(prev)}).",
    f"Entonces, el modelo ordena a los clientes mejor que el azar: en el {pct(auc)} de las parejas formadas por un "
    "cliente que suscribe y otro que no, asigna la probabilidad más alta al que suscribe. La discriminación es "
    "moderada y su intervalo de confianza del 95 %, calculado en el cuaderno (sección 4.4), va de 0,640 a 0,704, "
    "de modo que excluye 0,5.",
    "Por tal motivo, se toman el AUC y el AUC-PR como métricas principales y el modelo se presenta como una línea "
    "base con la que comparar modelos más flexibles en los siguientes entregables.")

**Interpretación.** En la prueba, el AUC-ROC es 0,677, el AUC-PR es 0,489 frente a una prevalencia de 0,308 (1,6 veces) y el error de Brier es 0,215 frente a 0,273 de la regla simple, una mejora relativa de 21,2 %. Entre los 8.236 contactos de prueba hay 2.539 suscripciones reales (30,8 %).

Entonces, el modelo ordena a los clientes mejor que el azar: en el 67,7 % de las parejas formadas por un cliente que suscribe y otro que no, asigna la probabilidad más alta al que suscribe. La discriminación es moderada y su intervalo de confianza del 95 %, calculado en el cuaderno (sección 4.4), va de 0,640 a 0,704, de modo que excluye 0,5.

Por tal motivo, se toman el AUC y el AUC-PR como métricas principales y el modelo se presenta como una línea base con la que comparar modelos más flexibles en los siguientes entregables.

## 3.2 Métricas en prueba de los tres modelos

In [4]:
tabla(metricas, dec=3)

,Exactitud,Precisión,Recall,F1,AUC-ROC,AUC-PR,Brier
Regla simple · siempre «No»,"0,692","0,000","0,000","0,000","0,500","0,308","0,273"
"Regresión logística · umbral 0,5","0,724","0,591","0,336","0,428","0,677","0,489","0,215"
"Regresión logística · umbral ajustado 0,031","0,342","0,312","0,942","0,469","0,677","0,489","0,215"


In [5]:
todos_f1 = 2 * prev / (1 + prev)           # F1 que se obtendría llamando a todos los clientes
assert m0["Exactitud"] > mu["Exactitud"] and m0["Recall"] == 0
interpretar(
    f"La regla simple logra una exactitud de {num(m0['Exactitud'], 3)} con precisión, recall y F1 de cero. La "
    f"regresión logística con umbral 0,5 llega a exactitud de {num(m05['Exactitud'], 3)}, precisión de "
    f"{num(m05['Precisión'], 3)}, recall de {num(m05['Recall'], 3)} y F1 de {num(m05['F1'], 3)}; con el umbral "
    f"ajustado ({num(umbral, 3)}), la exactitud es {num(mu['Exactitud'], 3)}, la precisión {num(mu['Precisión'], 3)}, "
    f"el recall {num(mu['Recall'], 3)} y el F1 {num(mu['F1'], 3)}. El AUC-ROC ({num(m05['AUC-ROC'], 3)}), el AUC-PR "
    f"({num(m05['AUC-PR'], 3)}) y el Brier ({num(m05['Brier'], 3)}) son iguales en las dos filas de la regresión "
    "porque no dependen del umbral.",
    f"Es decir, la exactitud no sirve para comparar: la regla simple supera en exactitud al modelo con el umbral "
    f"ajustado y, aun así, no encuentra a ningún suscriptor. Además, el F1 del umbral ajustado ({num(mu['F1'], 3)}) "
    f"es casi el que se obtendría llamando a todos los clientes ({num(todos_f1, 3)}), y su precisión "
    f"({num(mu['Precisión'], 3)}) apenas supera la prevalencia ({num(prev, 3)}).",
    f"Por ello, se opta por reportar ambos umbrales sin presentar el ajustado como una mejora: se aprendió con una "
    f"prevalencia de {pct(y_tr.mean())} y en la prueba pierde selectividad; se recomienda recalibrarlo con datos "
    "recientes en el siguiente entregable (secciones 4.3 y 4.4 del cuaderno).")

**Interpretación.** La regla simple logra una exactitud de 0,692 con precisión, recall y F1 de cero. La regresión logística con umbral 0,5 llega a exactitud de 0,724, precisión de 0,591, recall de 0,336 y F1 de 0,428; con el umbral ajustado (0,031), la exactitud es 0,342, la precisión 0,312, el recall 0,942 y el F1 0,469. El AUC-ROC (0,677), el AUC-PR (0,489) y el Brier (0,215) son iguales en las dos filas de la regresión porque no dependen del umbral.

Es decir, la exactitud no sirve para comparar: la regla simple supera en exactitud al modelo con el umbral ajustado y, aun así, no encuentra a ningún suscriptor. Además, el F1 del umbral ajustado (0,469) es casi el que se obtendría llamando a todos los clientes (0,471), y su precisión (0,312) apenas supera la prevalencia (0,308).

Por ello, se opta por reportar ambos umbrales sin presentar el ajustado como una mejora: se aprendió con una prevalencia de 6,4 % y en la prueba pierde selectividad; se recomienda recalibrarlo con datos recientes en el siguiente entregable (secciones 4.3 y 4.4 del cuaderno).

## 3.3 Curva ROC

In [6]:
mostrar(figura_con_titulo(comp["graficos"], "Curva ROC"), alto=420)

In [7]:
fpr, tpr, _ = roc_curve(y_te, p)
pt = {f: float(np.interp(f, fpr, tpr)) for f in (0.05, 0.10, 0.20, 0.50)}
interpretar(
    f"La curva queda por encima de la diagonal del azar, con un AUC de {num(auc, 3)}. Con 5 % de falsas alarmas se "
    f"detecta el {pct(pt[0.05])} de los suscriptores; con 10 %, el {pct(pt[0.10])}; con 20 %, el "
    f"{pct(pt[0.20])}; y con 50 %, el {pct(pt[0.50])}.",
    f"Esto quiere decir que el modelo es útil sobre todo en el tramo inicial, donde pocas falsas alarmas ya traen "
    f"detecciones, y que la curva se aplana hacia la mitad: pasar de 20 % a 50 % de falsas alarmas agrega solo "
    f"{num((pt[0.50] - pt[0.20]) * 100, 1)} puntos de detección.",
    "Por lo anterior, si la capacidad de llamadas es limitada, conviene priorizar por probabilidad y operar en el "
    "tramo inicial de la curva, fijando el punto de operación según el costo de cada llamada (sección 4.6 del "
    "cuaderno).")

**Interpretación.** La curva queda por encima de la diagonal del azar, con un AUC de 0,677. Con 5 % de falsas alarmas se detecta el 12,6 % de los suscriptores; con 10 %, el 32,5 %; con 20 %, el 48,9 %; y con 50 %, el 73,3 %.

Esto quiere decir que el modelo es útil sobre todo en el tramo inicial, donde pocas falsas alarmas ya traen detecciones, y que la curva se aplana hacia la mitad: pasar de 20 % a 50 % de falsas alarmas agrega solo 24,4 puntos de detección.

Por lo anterior, si la capacidad de llamadas es limitada, conviene priorizar por probabilidad y operar en el tramo inicial de la curva, fijando el punto de operación según el costo de cada llamada (sección 4.6 del cuaderno).

## 3.4 Matrices de confusión

In [8]:
mostrar(figura_con_titulo(comp["graficos"], "Matrices de confusión"), alto=460)
filas = {}
for nombre, pred in ((f"Umbral 0,5", r["prediction_05"]), (f"Umbral ajustado {num(umbral, 3)}", r["prediction_tuned"])):
    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
    filas[nombre] = {"Detectados": tp, "Perdidos": fn, "Falsas alarmas": fp, "Descartados bien": tn,
                     "Llamadas recomendadas": tp + fp, "Precisión": tp / (tp + fp), "Recall": tp / (tp + fn)}
detalle = pd.DataFrame(filas).T
tabla(detalle.assign(**{k: detalle[k].map(lambda v: num(v, 0)) for k in detalle.columns[:5]}), dec=3)

,Detectados,Perdidos,Falsas alarmas,Descartados bien,Llamadas recomendadas,Precisión,Recall
"Umbral 0,5",852,1.687,590,5.107,1.442,"0,591","0,336"
"Umbral ajustado 0,031",2.393,146,5.275,422,7.668,"0,312","0,942"


In [9]:
a, b = detalle.iloc[0], detalle.iloc[1]
reales, negativos = int(y_te.sum()), int((1 - y_te).sum())
interpretar(
    f"Con el umbral de 0,5, el modelo detecta {num(a['Detectados'], 0)} de los {num(reales, 0)} suscriptores "
    f"({pct(a['Recall'])}) y marca {num(a['Falsas alarmas'], 0)} falsas alarmas entre {num(negativos, 0)} clientes "
    f"que no suscriben ({pct(a['Falsas alarmas'] / negativos)}); de sus {num(a['Llamadas recomendadas'], 0)} "
    f"llamadas recomendadas, el {pct(a['Precisión'])} termina en suscripción. Con el umbral ajustado detecta "
    f"{num(b['Detectados'], 0)} ({pct(b['Recall'])}) a cambio de {num(b['Falsas alarmas'], 0)} falsas alarmas "
    f"({pct(b['Falsas alarmas'] / negativos)}); de sus {num(b['Llamadas recomendadas'], 0)} llamadas "
    f"recomendadas, el {pct(b['Precisión'])} termina en suscripción.",
    f"Entonces, con 0,5 la lista es corta y {num(a['Precisión'] / prev, 1)} veces más precisa que llamar al azar "
    f"({pct(a['Precisión'])} frente a {pct(prev)}), pero deja por fuera al {pct(1 - a['Recall'])} de los "
    f"suscriptores; con el umbral ajustado ocurre lo contrario: se llama al "
    f"{pct(b['Llamadas recomendadas'] / len(y_te))} de los clientes y la precisión apenas supera la tasa base.",
    "Por tal motivo, se decide elegir el punto de operación según la capacidad de llamadas y el costo y el "
    "beneficio de cada una, no con un umbral fijo (sección 4.5 del cuaderno).")

**Interpretación.** Con el umbral de 0,5, el modelo detecta 852 de los 2.539 suscriptores (33,6 %) y marca 590 falsas alarmas entre 5.697 clientes que no suscriben (10,4 %); de sus 1.442 llamadas recomendadas, el 59,1 % termina en suscripción. Con el umbral ajustado detecta 2.393 (94,2 %) a cambio de 5.275 falsas alarmas (92,6 %); de sus 7.668 llamadas recomendadas, el 31,2 % termina en suscripción.

Entonces, con 0,5 la lista es corta y 1,9 veces más precisa que llamar al azar (59,1 % frente a 30,8 %), pero deja por fuera al 66,4 % de los suscriptores; con el umbral ajustado ocurre lo contrario: se llama al 93,1 % de los clientes y la precisión apenas supera la tasa base.

Por tal motivo, se decide elegir el punto de operación según la capacidad de llamadas y el costo y el beneficio de cada una, no con un umbral fijo (sección 4.5 del cuaderno).

## 3.5 Lectura del punto operativo

In [10]:
cuantiles = pd.DataFrame({"Percentil": ["5", "25", "50", "75", "95"],
                          "Probabilidad predicha": np.quantile(p, [0.05, 0.25, 0.50, 0.75, 0.95])}).set_index("Percentil")
tabla(cuantiles, dec=3)

,Probabilidad predicha
Percentil,
5,"0,028"
25,"0,053"
50,"0,090"
75,"0,353"
95,"0,928"


In [11]:
interpretar(
    f"Con el umbral de 0,5 el modelo marca como posibles suscriptores al {pct(r['prediction_05'].mean())} de la "
    f"prueba y con el umbral ajustado ({num(umbral, 3)}), al {pct(r['prediction_tuned'].mean())}. La mediana de las "
    f"probabilidades predichas es {num(np.median(p), 3)}, el percentil 75 es {num(np.quantile(p, 0.75), 3)} y el 5 % "
    f"más alto supera {num(np.quantile(p, 0.95), 3)}.",
    f"Esto significa que las probabilidades están desplazadas hacia abajo respecto de la tasa real: la media "
    f"predicha es {num(p.mean(), 3)} y la tasa observada {num(prev, 3)}, porque el modelo aprendió con una "
    f"prevalencia de {pct(y_tr.mean())}. Un umbral de {num(umbral, 3)} queda por debajo de la probabilidad del "
    f"{pct(r['prediction_tuned'].mean())} de los clientes, de modo que apenas filtra.",
    "Así las cosas, se decide no leer las probabilidades como frecuencias esperadas y dejar la recalibración del "
    "intercepto con datos recientes como siguiente paso (sección 4.7 del cuaderno).")

**Interpretación.** Con el umbral de 0,5 el modelo marca como posibles suscriptores al 17,5 % de la prueba y con el umbral ajustado (0,031), al 93,1 %. La mediana de las probabilidades predichas es 0,090, el percentil 75 es 0,353 y el 5 % más alto supera 0,928.

Esto significa que las probabilidades están desplazadas hacia abajo respecto de la tasa real: la media predicha es 0,243 y la tasa observada 0,308, porque el modelo aprendió con una prevalencia de 6,4 %. Un umbral de 0,031 queda por debajo de la probabilidad del 93,1 % de los clientes, de modo que apenas filtra.

Así las cosas, se decide no leer las probabilidades como frecuencias esperadas y dejar la recalibración del intercepto con datos recientes como siguiente paso (sección 4.7 del cuaderno).

## 3.6 Configuración del modelo

In [12]:
tss = TimeSeriesSplit(n_splits=d.VALIDATION_SPLITS, gap=r["validation_gap"])
prev_val = [float(y_tr.iloc[val].mean()) for _, val in tss.split(r["x_train"])]
f1_todos_val = float(np.mean([2 * q / (1 + q) for q in prev_val]))
configuracion = pd.DataFrame({
    "Elemento": ["Regularización", "Variables numéricas", "Variables categóricas", "Variables excluidas",
                 "Validación temporal", "Separación entre ajuste y validación", "Umbral ajustado"],
    "Valor": [f"C = {d.LOGISTIC_C:g}",
              "`age`, `cons.price.idx`, `cons.conf.idx` y `nr.employed` escaladas; `campaign` y `previous` con `log1p`; "
              "`pdays` con indicador de 999",
              f"one-hot, con las categorías de menos del {pct(d.MIN_CATEGORY_FREQUENCY)} agrupadas como infrecuentes",
              "`duration` (fuga), `euribor3m` y `emp.var.rate` (redundancia)",
              f"{d.VALIDATION_SPLITS} pliegues, con prevalencias de {pct(min(prev_val))} a {pct(max(prev_val))}",
              f"{num(r['validation_gap'], 0)} registros (percentil {num(d.GAP_DAY_QUANTILE * 100, 0)} de registros por día de contacto)",
              f"{num(umbral, 3)}: maximiza el F1 promedio de validación ({num(r['validation_f1'], 3)})"],
})
tabla(configuracion, indice=False)

Elemento,Valor
Regularización,C = 100
Variables numéricas,"`age`, `cons.price.idx`, `cons.conf.idx` y `nr.employed` escaladas; `campaign` y `previous` con `log1p`; `pdays` con indicador de 999"
Variables categóricas,"one-hot, con las categorías de menos del 0,5 % agrupadas como infrecuentes"
Variables excluidas,"`duration` (fuga), `euribor3m` y `emp.var.rate` (redundancia)"
Validación temporal,"5 pliegues, con prevalencias de 3,9 % a 14,0 %"
Separación entre ajuste y validación,507 registros (percentil 95 de registros por día de contacto)
Umbral ajustado,"0,031: maximiza el F1 promedio de validación (0,131)"


In [13]:
interpretar(
    f"El modelo usa regularización débil (C = {d.LOGISTIC_C:g}), transformación `log1p` en `campaign` y `previous`, "
    f"codificación one-hot y un indicador para `pdays` = 999, y excluye `duration`, `euribor3m` y `emp.var.rate`. El "
    f"umbral ajustado, {num(umbral, 3)}, maximiza el F1 promedio de {d.VALIDATION_SPLITS} pliegues temporales "
    f"separados por {num(r['validation_gap'], 0)} registros, y ese F1 es {num(r['validation_f1'], 3)}.",
    f"Hay que tener en cuenta que la separación evita que el pliegue de validación comparta días de contacto con el "
    f"de ajuste, y que el F1 de validación ({num(r['validation_f1'], 3)}) es mucho menor que el de la prueba "
    f"({num(mu['F1'], 3)}) en buena medida porque la prevalencia de los pliegues es baja, de "
    f"{pct(min(prev_val))} a {pct(max(prev_val))}. En esos pliegues, llamar a todos los clientes daría un F1 promedio de "
    f"{num(f1_todos_val, 3)}, de modo que el umbral aprendido apenas mejora esa referencia.",
    "Por tal motivo, se mantiene este esquema temporal como referencia y las métricas de validación se leen como una "
    "estimación conservadora y variable del desempeño futuro (secciones 4.1 a 4.3 del cuaderno).")

**Interpretación.** El modelo usa regularización débil (C = 100), transformación `log1p` en `campaign` y `previous`, codificación one-hot y un indicador para `pdays` = 999, y excluye `duration`, `euribor3m` y `emp.var.rate`. El umbral ajustado, 0,031, maximiza el F1 promedio de 5 pliegues temporales separados por 507 registros, y ese F1 es 0,131.

Hay que tener en cuenta que la separación evita que el pliegue de validación comparta días de contacto con el de ajuste, y que el F1 de validación (0,131) es mucho menor que el de la prueba (0,469) en buena medida porque la prevalencia de los pliegues es baja, de 3,9 % a 14,0 %. En esos pliegues, llamar a todos los clientes daría un F1 promedio de 0,130, de modo que el umbral aprendido apenas mejora esa referencia.

Por tal motivo, se mantiene este esquema temporal como referencia y las métricas de validación se leen como una estimación conservadora y variable del desempeño futuro (secciones 4.1 a 4.3 del cuaderno).

## Dónde ampliar lo que el dashboard no muestra

| Detalle | Dónde se desarrolla |
|---|---|
| Búsqueda de `C` con validación temporal y comparación con la línea base trivial | {doc}`secciones 4.1 y 4.2 del cuaderno <../desarrollo/04_modelo-base>` |
| Elección del umbral de decisión por pliegue | {doc}`sección 4.3 <../desarrollo/04_modelo-base>` |
| Intervalos de confianza *bootstrap* por bloques de día y diferencias frente a la línea base | {doc}`sección 4.4 <../desarrollo/04_modelo-base>` |
| Curva precisión-recall y calibración de las probabilidades | {doc}`secciones 4.6 y 4.7 <../desarrollo/04_modelo-base>` |
| Curva de aprendizaje | {doc}`sección 4.8 <../desarrollo/04_modelo-base>` |
| Interpretación de los coeficientes (*odds ratios*) | {doc}`sección 4.9 <../desarrollo/04_modelo-base>` |
| Diagnóstico temporal: validación aleatoria de contraste, desempeño por año y residuos | {doc}`sección 4.10 <../desarrollo/04_modelo-base>` |
| Recorrido de la nota crítica del enunciado, conclusiones y limitaciones | {doc}`secciones 4.11 y 5 <../desarrollo/05_conclusiones-y-limitaciones>` |